In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="ylacombe/google-colombian-spanish",
    repo_type="dataset", local_dir="./google-colombian-spanish", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|██████████| 6/6 [00:02<00:00,  2.01it/s]


'/home/ubuntu/google-colombian-spanish'

In [3]:
files = glob('google-colombian-spanish/*/*.parquet')
len(files)

6

In [4]:
df = pd.read_parquet(files[0])
df.head()

,audio,text,speaker_id
0,{'bytes': b'RIFF$\x00\x06\x00WAVEfmt \x10\x00\...,¿Cuál es la estación de metro más cercana?,2484
1,{'bytes': b'RIFF$\xa0\x05\x00WAVEfmt \x10\x00\...,Siempre he sido muy fan del color gris claro,9697
2,{'bytes': b'RIFF$\xe0\t\x00WAVEfmt \x10\x00\x0...,¿Te gustan los chocolates?,6136
3,{'bytes': b'RIFF$\x80\x04\x00WAVEfmt \x10\x00\...,¿A qué hora cierra el hospital veterinario?,1523
4,{'bytes': b'RIFF$`\x08\x00WAVEfmt \x10\x00\x00...,¿Qué es lo que necesitas?,6136


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 789/789 [01:03<00:00, 12.48it/s]


In [7]:
len(data)

4903

In [8]:
data[0]

{'audio_filename': 'google-colombian-spanish_audio/google-colombian-spanish-male-train-00002-of-00003-c064286c37b4b24a_0.mp3',
 'text': '¿Cuál es la estación de metro más cercana?',
 'speaker': 'google-colombian-spanish_audio_2484'}

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('google-colombian-spanish-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'google-colombian-spanish_audio/google-colombian-spanish-male-train-00002-of-00003-c064286c37b4b24a_0.mp3',
 'text': '¿Cuál es la estación de metro más cercana?',
 'speaker': 'google-colombian-spanish_audio_2484'}

In [11]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'google-colombian-spanish')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 502.73ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  152kB /  152kB,   ???B/s  
Processing Files (1 / 1): 100%|██████████|  152kB /  152kB,  0.00B/s  
New Data Upload: 100%|██████████|  152kB /  152kB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  3.16 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/3cb0890804df6adb18d35e3b8112bdb55b9af59e', commit_message='Upload dataset', commit_description='', oid='3cb0890804df6adb18d35e3b8112bdb55b9af59e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
# !zip -rq google-colombian-spanish_audio.zip google-colombian-spanish_audio

In [15]:
# !hf upload malaysia-ai/Multilingual-TTS google-colombian-spanish_audio.zip --repo-type=dataset